In [ ]:
# Change this to your preferred framework (e.g., 'cuda', 'pytorch', 'triton', 'jax', 'mojo')
EVAL_LANG = 'cuda'

SAVE_GPU = True


<p>
  Given an array of <code>N</code> 32-bit floating point <code>values</code> and an integer array
  <code>flags</code> of the same length, where <code>flags[i] = 1</code> marks the start of a new
  segment and <code>flags[i] = 0</code> continues the current segment, compute the
  <strong>exclusive prefix sum within each segment</strong> and store the result in
  <code>output</code>. The first element is always a segment start
  (<code>flags[0] = 1</code>). Within each segment, <code>output[i]</code> equals the sum of all
  <code>values</code> elements in the same segment that appear before index <code>i</code>, so the
  first element of every segment is always <code>0.0</code>.
</p>

<h2>Implementation Requirements</h2>
<ul>
  <li>Use only native features (external libraries are not permitted)</li>
  <li>The <code>solve</code> function signature must remain unchanged</li>
  <li>The final result must be stored in <code>output</code></li>
  <li>Read from <code>values</code> and <code>flags</code>; write to <code>output</code></li>
</ul>

<h2>Example</h2>
<pre>
Input values: [1.0, 2.0, 3.0, 4.0, 5.0, 6.0]
Input flags:  [  1,   0,   0,   1,   0,   1]

Segments:     [1.0, 2.0, 3.0] | [4.0, 5.0] | [6.0]

Output:       [0.0, 1.0, 3.0,   0.0, 4.0,   0.0]
</pre>
<p>
  Segment 1: exclusive prefix sums of [1, 2, 3] &rarr; [0, 1, 3]<br>
  Segment 2: exclusive prefix sums of [4, 5] &rarr; [0, 4]<br>
  Segment 3: exclusive prefix sums of [6] &rarr; [0]
</p>

<h2>Constraints</h2>
<ul>
  <li>1 &le; <code>N</code> &le; 100,000,000</li>
  <li><code>flags[0] = 1</code> always (the first element starts the first segment)</li>
  <li><code>flags[i]</code> &isin; {0, 1} for all <code>i</code></li>
  <li>Values are 32-bit floats in the range [-100, 100]</li>
  <li>Performance is measured with <code>N</code> = 50,000,000</li>
</ul>


# CUDA

In [ ]:
%%writefile solution.cu
#include <cuda_runtime.h>

// values, flags, output are device pointers
extern "C" void solve(const float* values, const int* flags, float* output, int N) {}


# CUTE

In [ ]:
%%writefile solution.py
import cutlass
import cutlass.cute as cute


# values, flags, output are tensors on the GPU
@cute.jit
def solve(values: cute.Tensor, flags: cute.Tensor, output: cute.Tensor, N: cute.Int32):
    pass


# JAX

In [ ]:
%%writefile solution.py
import jax
import jax.numpy as jnp


# values, flags are tensors on the GPU
@jax.jit
def solve(values: jax.Array, flags: jax.Array, N: int) -> jax.Array:
    # return output tensor directly
    pass


# MOJO

In [ ]:
%%writefile solution.mojo
from std.gpu.host import DeviceContext
from std.gpu import block_dim, block_idx, thread_idx
from std.memory import UnsafePointer
from std.math import ceildiv


# values, flags, output are device pointers
@export
def solve(
    values: UnsafePointer[Float32, MutExternalOrigin],
    flags: UnsafePointer[Int32, MutExternalOrigin],
    output: UnsafePointer[Float32, MutExternalOrigin],
    N: Int32,
) raises:
    pass


# Torch

In [ ]:
%%writefile solution.py
import torch


# values, flags, output are tensors on the GPU
def solve(values: torch.Tensor, flags: torch.Tensor, output: torch.Tensor, N: int):
    pass


# Triton

In [ ]:
%%writefile solution.py
import torch
import triton
import triton.language as tl


# values, flags, output are tensors on the GPU
def solve(values: torch.Tensor, flags: torch.Tensor, output: torch.Tensor, N: int):
    pass


# Evaluate Setup

In [ ]:
# --- Core Challenge Base ---
from abc import ABC, abstractmethod
from typing import Any, Dict, List


class ChallengeBase(ABC):
    def __init__(self, name: str, atol: float, rtol: float, num_gpus: int, access_tier: str):
        self.name = name
        self.atol = atol
        self.rtol = rtol
        self.num_gpus = num_gpus
        self.access_tier = access_tier

    @abstractmethod
    def reference_impl(self, *args, **kwargs):
        """
        Reference solution implementation.
        """
        pass

    @abstractmethod
    def get_solve_signature(self) -> Dict[str, Any]:
        """
        Get the function signature for solution.

        Returns:
            Dictionary with argtypes and restype for ctypes
        """
        pass

    @abstractmethod
    def generate_example_test(self) -> List[Dict[str, Any]]:
        """
        Generate an example test case for this problem.

        Returns:
            Dictionary with test case parameters
        """
        pass

    @abstractmethod
    def generate_functional_test(self) -> List[Dict[str, Any]]:
        """
        Generate functional test cases for this problem.

        Returns:
            List of test case dictionaries
        """
        pass

    @abstractmethod
    def generate_performance_test(self) -> List[Dict[str, Any]]:
        """
        Generate a performance test case for this problem.

        Returns:
            Dictionary with test case parameters
        """
        pass


# --- Challenge Logic ---
import ctypes
from typing import Any, Dict, List

import torch


class Challenge(ChallengeBase):
    def __init__(self):
        super().__init__(
            name="Segmented Exclusive Prefix Sum",
            atol=1e-03,
            rtol=1e-03,
            num_gpus=1,
            access_tier="free",
        )

    def reference_impl(
        self,
        values: torch.Tensor,
        flags: torch.Tensor,
        output: torch.Tensor,
        N: int,
    ):
        assert values.shape == (N,)
        assert flags.shape == (N,)
        assert output.shape == (N,)
        assert values.dtype == torch.float32
        assert flags.dtype == torch.int32
        assert values.device.type == "cuda"

        # Global exclusive prefix sum (use float64 for accuracy in reference).
        excl = torch.empty(N, dtype=torch.float64, device="cuda")
        excl[0] = 0.0
        if N > 1:
            excl[1:] = torch.cumsum(values[:-1].double(), dim=0)

        # The exclusive prefix sum within each segment equals the global exclusive
        # prefix sum minus the global exclusive prefix sum at the segment start.
        # Use segment IDs (0-indexed) to index the per-segment offsets.
        seg_ids = torch.cumsum(flags.long(), dim=0) - 1  # segment index for each element
        seg_mask = flags.bool()
        # excl value at each segment start
        seg_start_excl = excl[seg_mask]  # shape: (num_segments,)
        # Broadcast segment start offset to every element in that segment
        per_elem_offset = seg_start_excl[seg_ids]

        output.copy_((excl - per_elem_offset).float())

    def get_solve_signature(self) -> Dict[str, tuple]:
        return {
            "values": (ctypes.POINTER(ctypes.c_float), "in"),
            "flags": (ctypes.POINTER(ctypes.c_int), "in"),
            "output": (ctypes.POINTER(ctypes.c_float), "out"),
            "N": (ctypes.c_int, "in"),
        }

    def generate_example_test(self) -> Dict[str, Any]:
        dtype_f = torch.float32
        dtype_i = torch.int32
        # Three segments: [1,2,3], [4,5], [6]
        # exclusive prefix sums: [0,1,3], [0,4], [0]
        values = torch.tensor([1.0, 2.0, 3.0, 4.0, 5.0, 6.0], device="cuda", dtype=dtype_f)
        flags = torch.tensor([1, 0, 0, 1, 0, 1], device="cuda", dtype=dtype_i)
        output = torch.empty(6, device="cuda", dtype=dtype_f)
        return {
            "values": values,
            "flags": flags,
            "output": output,
            "N": 6,
        }

    def generate_functional_test(self) -> List[Dict[str, Any]]:
        dtype_f = torch.float32
        dtype_i = torch.int32
        tests = []

        def make_test(vals, segs):
            """vals: list of floats, segs: list of segment start indices"""
            N = len(vals)
            flags = torch.zeros(N, dtype=dtype_i)
            for s in segs:
                flags[s] = 1
            return {
                "values": torch.tensor(vals, device="cuda", dtype=dtype_f),
                "flags": flags.cuda(),
                "output": torch.empty(N, device="cuda", dtype=dtype_f),
                "N": N,
            }

        def make_random_test(N, avg_seg_len, seed=None):
            if seed is not None:
                torch.manual_seed(seed)
            vals = torch.empty(N, dtype=dtype_f).uniform_(-10.0, 10.0)
            flags = torch.zeros(N, dtype=dtype_i)
            flags[0] = 1
            i = avg_seg_len
            while i < N:
                flags[i] = 1
                i += max(1, int(torch.randint(1, 2 * avg_seg_len + 1, (1,)).item()))
            return {
                "values": vals.cuda(),
                "flags": flags.cuda(),
                "output": torch.empty(N, device="cuda", dtype=dtype_f),
                "N": N,
            }

        # Edge: single element, single segment
        tests.append(make_test([5.0], [0]))

        # Edge: two elements, one segment
        tests.append(make_test([3.0, 7.0], [0]))

        # Edge: two elements, two segments
        tests.append(make_test([3.0, 7.0], [0, 1]))

        # Edge: four elements, all in one segment
        tests.append(make_test([1.0, 2.0, 3.0, 4.0], [0]))

        # Four elements, each its own segment (all outputs = 0)
        tests.append(make_test([1.0, -2.0, 3.0, -4.0], [0, 1, 2, 3]))

        # Negative values in mixed segments: two segments of length 3
        tests.append(make_test([-1.0, -2.0, -3.0, 5.0, 6.0, -7.0], [0, 3]))

        # Power-of-2: N=16, two equal segments
        tests.append(make_test([float(i) for i in range(16)], [0, 8]))

        # Power-of-2: N=32, segments of length 4
        tests.append(make_test([1.0] * 32, list(range(0, 32, 4))))

        # Power-of-2: N=64, random segment lengths ~8
        tests.append(make_random_test(64, avg_seg_len=8, seed=42))

        # Power-of-2: N=128, random segment lengths ~16
        tests.append(make_random_test(128, avg_seg_len=16, seed=7))

        # Non-power-of-2: N=30, segments of length ~5
        tests.append(make_random_test(30, avg_seg_len=5, seed=13))

        # Non-power-of-2: N=100, small segments of length ~3
        tests.append(make_random_test(100, avg_seg_len=3, seed=99))

        # Non-power-of-2: N=255, segments spanning multiple warps
        tests.append(make_random_test(255, avg_seg_len=32, seed=17))

        # Realistic: N=1024, segments of length ~64
        tests.append(make_random_test(1024, avg_seg_len=64, seed=11))

        # Realistic: N=10000, segments crossing block boundaries
        tests.append(make_random_test(10000, avg_seg_len=256, seed=55))

        return tests

    def generate_performance_test(self) -> Dict[str, Any]:
        dtype_f = torch.float32
        dtype_i = torch.int32
        N = 50_000_000
        torch.manual_seed(42)
        vals = torch.empty(N, dtype=dtype_f).uniform_(-1.0, 1.0)
        flags = torch.zeros(N, dtype=dtype_i)
        flags[0] = 1
        # Segments of average length 256 (crosses many thread blocks)
        seg_starts = torch.arange(256, N, 256, dtype=torch.long)
        flags[seg_starts] = 1
        return {
            "values": vals.cuda(),
            "flags": flags.cuda(),
            "output": torch.empty(N, device="cuda", dtype=dtype_f),
            "N": N,
        }


ch = Challenge()



In [ ]:
import os
import time
import ctypes
import torch

class Evaluate:
    @staticmethod
    def eval_cuda(ch):
        # 1. Compile a fresh uniquely named library
        so_filename = f'solution_func_{int(time.time())}.so'
        os.system(f'nvcc -shared -Xcompiler -fPIC -O3 solution.cu -o {so_filename}')
        lib = ctypes.CDLL(f'./{so_filename}')
        
        # 2. Extract signature and set argtypes
        signature = ch.get_solve_signature()
        lib.solve.argtypes = [arg_info[0] for arg_info in signature.values()]
        
        Evaluate._run_tests(ch, signature, lambda kwargs: lib.solve(*Evaluate._build_cuda_args(kwargs, signature)))

    @staticmethod
    def eval_python(ch):
        import importlib.util
        import sys
        
        spec = importlib.util.spec_from_file_location("solution", "solution.py")
        solution = importlib.util.module_from_spec(spec)
        sys.modules["solution"] = solution
        spec.loader.exec_module(solution)
        
        signature = ch.get_solve_signature()
        Evaluate._run_tests(ch, signature, lambda kwargs: Evaluate._run_python(solution, kwargs))

    @staticmethod
    def _run_python(solution, kwargs):
        solution.solve(**kwargs)
        if torch.cuda.is_available():
            torch.cuda.synchronize()

    @staticmethod
    def eval_mojo(ch):
        print("Mojo evaluation is currently executed via a separate runner or wrapper.")
        print("Ensure you have the mojo compiler installed and use 'mojo build solution.mojo' + ctypes/ffi,")
        print("or run an external python bridge. This is a stub.")

    @staticmethod
    def _build_cuda_args(kwargs, signature):
        cuda_args = []
        for k, (arg_type, dir_type) in signature.items():
            val = kwargs[k]
            if isinstance(val, torch.Tensor):
                cuda_args.append(ctypes.cast(val.data_ptr(), arg_type))
            else:
                cuda_args.append(arg_type(val))
        return cuda_args

    @staticmethod
    def _run_tests(ch, signature, run_fn):
        print("=== Running Functional Tests ===")
        functional_tests = ch.generate_functional_test()
        all_passed = True
        
        for i, test in enumerate(functional_tests):
            ref_kwargs = {k: (v.clone() if isinstance(v, torch.Tensor) else v) for k, v in test.items()}
            test_kwargs = {k: (v.clone() if isinstance(v, torch.Tensor) else v) for k, v in test.items()}
            
            # Run Reference
            ch.reference_impl(**ref_kwargs)
            
            # Run implementation
            run_fn(test_kwargs)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            
            # Verify outputs
            match = True
            for k, (_, dir_type) in signature.items():
                if dir_type == "out":
                    if not torch.allclose(ref_kwargs[k], test_kwargs[k], atol=ch.atol, rtol=ch.rtol):
                        match = False
                        print(f"❌ Test {i+1}/{len(functional_tests)} Failed on output '{k}'")
                        break
            
            if match:
                print(f"✅ Test {i+1}/{len(functional_tests)} Passed")
            else:
                all_passed = False
                break
                
        if all_passed:
            print("\n🎉 All functional tests passed!")
            return True
        else:
            return False



# Evaluation code

In [ ]:
# Run the evaluator based on configuration
if EVAL_LANG == 'cuda':
    Evaluate.eval_cuda(ch)
elif EVAL_LANG in ['pytorch', 'triton', 'jax', 'cute']:
    Evaluate.eval_python(ch)
elif EVAL_LANG == 'mojo':
    Evaluate.eval_mojo(ch)
else:
    print(f"Unknown language {EVAL_LANG}")

# Disconnect runtime to save Colab resources
if SAVE_GPU:
    from google.colab import runtime
    runtime.unassign()
